# 04 · Measure the increment and decide what follows


A valid negative result is useful. A missing prediction, a corrupted mask and a
complete failed effect criterion are different outcomes. Execution reconstructs
and scores the saved models before sealing a result. Inspect mode reads the
saved evidence without rewriting it.

**Run this notebook independently in a fresh kernel.** The default
`teach` mode uses small generated examples. Set `FI_TUTORIAL_MODE=inspect`
and `FI_RUN_ROOT` before starting the kernel to read saved artifacts.
Set `FI_TUTORIAL_MODE=execute` with an explicit `FI_RUN_ROOT` to run the
production stages below. Execute notebooks **00 → 04** in order for the
full Experiment 0; each uses a fresh kernel and the same run directory.
Use the [notebook HAIC launchers](../../../slurm/future-innovation/NOTEBOOKS.md)
for scheduled execution. Inspection remains read-only. An absent local
file says nothing about the current state of a remote HAIC job.

[Study overview](../../../docs/studies/future-innovation/README.md) ·
[Historical direct-v2 specification](../../../docs/studies/future-innovation/direct-gate-protocol.md) ·
[Calibrated direct-v3 specification](../../../docs/studies/future-innovation/direct-v3-repair-protocol.md)

In [ ]:
from pathlib import Path
import os
import sys
from time import perf_counter

started = perf_counter()
override = os.environ.get("GAVD6_ROOT")
if override:
    candidates = [Path(override).expanduser().resolve()]
else:
    candidates = []
    for base in (Path.cwd(), *Path.cwd().parents):
        candidates.extend((base, base / "gavd6", base / "experiments/sjepa/gavd6"))
PROJECT_ROOT = next((p for p in candidates if (p / "src/gavd6_sjepa").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Set GAVD6_ROOT to the checkout containing src/gavd6_sjepa.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from matplotlib_inline.backend_inline import set_matplotlib_formats
get_ipython().run_line_magic("matplotlib", "inline")
set_matplotlib_formats("svg", "png")
plt.rcParams.update({"figure.figsize": (8, 3), "axes.spines.top": False,
                    "axes.spines.right": False, "font.size": 11})

from gavd6_sjepa.research_directions.future_innovation.fi_tutorial_inspection import (
    artifact_inventory, inspect_report, read_optional_table, inspection_audit_path,
)

# Use "execute" for real stages or "inspect" for saved artifacts.
# Relative paths resolve from GAVD6_ROOT. Execution requires an explicit run root.
MODE = os.environ.get("FI_TUTORIAL_MODE", "teach")
if MODE not in {"teach", "inspect", "execute"}:
    raise ValueError("FI_TUTORIAL_MODE must be teach, inspect, or execute.")
if MODE == "execute" and not os.environ.get("FI_RUN_ROOT"):
    raise ValueError("Set FI_RUN_ROOT explicitly before executing real experiment stages.")
RUN_ROOT = Path(os.environ.get("FI_RUN_ROOT", "outputs/future-innovation-direct-v3-dev-20260911")).expanduser()
if not RUN_ROOT.is_absolute():
    RUN_ROOT = PROJECT_ROOT / RUN_ROOT
RUN_ROOT = RUN_ROOT.resolve()
print("Teaching examples only; no empirical gait findings." if MODE == "teach"
      else f"{MODE.upper()} mode: {RUN_ROOT}")
if MODE == "execute":
    from gavd6_sjepa.research_directions.future_innovation.fi_notebook_workflow import (
        initialize_from_environment, run_stage, build_notebook_report, finish_notebook_report,
        attempt_stage, require_stage_success,
    )
    if (RUN_ROOT / "config/run-contract.json").is_file():
        import json
        saved_run = json.loads((RUN_ROOT / "config/run-contract.json").read_text())
        print("Frozen protocol:", saved_run.get("protocol", "legacy-v1"),
              "— gate clips:", saved_run.get("cohort_size"))
        if saved_run.get("protocol", "legacy-v1") == "legacy-v1":
            print("This run retains legacy selectivity gates. Use a new run root for direct-v2.")

## Execute this stage

Score the out-of-fold predictions and build the sealed production report. If scoring fails, still attempt a diagnostic STOP report. A verified audit rejection builds an unsealed diagnostic STOP and finishes with TRAINING BLOCKED, measurement_complete=False. Unexpected scoring failures and missing or corrupt evidence still fail. Complete STOP and INCONCLUSIVE results are also successful executions.

This cell runs only in `execute` mode. Each command uses this kernel's Python and the existing production CLI; stage logs are retained alongside the executed notebook.

In [ ]:
if MODE == "execute":
    scoring_succeeded = build_notebook_report(RUN_ROOT)

Pool held-out predictions over the five folds. For each target feature, R² is
one minus source-weighted squared prediction error divided by the error of its
outer-training mean. Average featurewise R² on the intersection of training-valid
dimensions. Stochastic historical runs then average seed scores; deterministic
direct-v3 has one score. Neither fold-score averaging nor test-mean centering
implements this metric.

The 2,000 paired bootstrap draws resample whole source videos with replacement.
Keep clips together and count a source twice when drawn twice. The same draws
serve every arm. These intervals condition on saved models and omit repeated
fitting, selection and this cohort's adaptive redesign. They add no independent
sources. Report the 95% interval separately from the 90%-positive decision rule.

| Required criterion | Threshold |
|---|---|
| Mean real gain over shared RGB ridge | at least +0.05 R² |
| Shuffle | real gain ≥ 2 × max(shuffled gain, 0) |
| Mismatch | gain ≤ +0.01 R² |
| Matched skeleton increment | real minus no-skeleton > 0 |
| Paired bootstrap | real gain and matched increment each positive in ≥90% of draws |
| Measurement evidence | all required input, target, audit, source, model and numerical checks valid |

Direct-v3's prospective deterministic policy makes stochastic seed stability
inapplicable. Historical stochastic runs still require all three gains positive,
at least two ≥0.05, and matched increment positive in every seed. Point failures
produce complete STOP. Passed points with insufficient stability produce
INCONCLUSIVE. A passing direct-v3 result is development ADVANCE and requires
independent-source confirmation before scaling.

In [ ]:
if MODE=='teach':
    from gavd6_sjepa.research_directions.future_innovation.fi_joint_reporting import decide_joint_gate
    metrics={'delta_r2_real':.06,'delta_r2_time_shuffle':.02,'delta_r2_clip_mismatch':.005,'delta_r2_no_skeleton':.005,
             'seed_real_gains':[.06],'seed_skeleton_increments':[.055],
             'bootstrap_positive_fraction':.95,'skeleton_increment_positive_fraction':.95,
             **{k:True for k in ('data_contract_valid','evaluation_contract_valid','controls_complete','input_audit_complete',
                                  'target_variance_valid','teacher_stable','causal_leakage_absent')}}
    cases={'Invented passing development evidence':metrics,
           'Invented uncertainty':{**metrics,'bootstrap_positive_fraction':.7},
           'Exact fallback / zero increment':{**metrics,'delta_r2_real':0.,'delta_r2_no_skeleton':0.,'seed_real_gains':[0.],'seed_skeleton_increments':[0.]}}
    display(pd.DataFrame([{'synthetic illustration':name,**{k:decide_joint_gate(m)[k] for k in ('decision','allow_full_experiment','stochastic_seed_stability')}} for name,m in cases.items()]))

In [ ]:
if MODE!='teach':
    evidence=inspect_report(RUN_ROOT)
    print(evidence['state'],evidence['explanation'])
    print('Report seal integrity checked:',evidence['seal_verified'])
    for path in ['reports/aggregate-metrics.csv','reports/paired-controls.csv']:
        table=read_optional_table(RUN_ROOT,path)
        if table is not None:
            with pd.option_context('display.precision',10,'display.max_columns',None): display(table)
    from gavd6_sjepa.research_directions.future_innovation.fi_contracts import read_json
    for path in ['reports/uncertainty.json','reports/numerical-verification.json']:
        p=RUN_ROOT/path
        if p.is_file():
            print(path,'— saved evidence; numerical reconstruction is not rerun in inspect mode')
            display(read_json(p))
    aggregate=read_optional_table(RUN_ROOT,'reports/aggregate-metrics.csv')
    if aggregate is not None:
        points=aggregate.groupby('arm',sort=False).delta_r2.mean()
        fig,ax=plt.subplots(figsize=(9,3)); ax.barh(points.index,points.values,color='#2f6f99')
        ax.axvline(0,color='black',linewidth=.7); ax.axvline(.05,color='#d4803f',linestyle='--',label='Required real gain')
        ax.set(xlabel='Gain over shared RGB reference (R²)'); ax.legend(); plt.show()
    if evidence['report_text'] is not None: display(Markdown(evidence['report_text']))

A negative repaired result concerns this predictor, these ordered summaries,
the contextual teacher target and the small data regime. A separately specified
representation, target or data study may be reasonable, but this result does
not justify increasing student capacity by itself. A positive development result
requires independent-source confirmation. Neither outcome trains S-JEPA,
adapters or a skeleton-only distilled student; those remain subsequent experiments.

In [ ]:
if MODE == "execute":
    completed_decision = finish_notebook_report(RUN_ROOT, scoring_succeeded=scoring_succeeded)

## What this step establishes

Record the exact run, measured effect, uncertainty and evidence status. A complete negative measurement closes this comparison; an incomplete result identifies required recovery work.

Return to the [study overview](../../../docs/studies/future-innovation/README.md) to record the next decision.

In [ ]:
print(f"Notebook elapsed time: {perf_counter() - started:.2f} seconds ({MODE} mode).")